In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gpreda/medquad/medquad.csv


In [2]:
 !pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.5 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.2

In [3]:
 !pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.5 MB/s eta 0:00:00


In [4]:
import sentence_transformers
import transformers
import torch 
import pandas as pd
import numpy as np

In [5]:
df=pd.read_csv('/kaggle/input/datasets/gpreda/medquad/medquad.csv')
print(df.shape)
print(df.columns.tolist())
df.head()

(16412, 4)
['question', 'answer', 'source', 'focus_area']


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [6]:
print(df.isnull().sum())
print(df.duplicated(subset=['question','answer']).sum())

df['answer_len'] = df['answer'].str.len()
print(df['answer_len'].describe())

question       0
answer         5
source         0
focus_area    14
dtype: int64
48
count    16407.000000
mean      1303.452673
std       1656.694326
min          6.000000
25%        487.000000
50%        890.000000
75%       1589.000000
max      29046.000000
Name: answer_len, dtype: float64


In [7]:
df=df.dropna(subset=['answer']).drop_duplicates(subset=['question','answer']).reset_index(drop=True)
print(df.shape)

(16359, 5)


In [8]:
def fixed_chunks(text,chunk_size=500,overlap=50):
    chunks=[]
    start=0
    while start<len(text):
        end=start+chunk_size
        chunks.append(text[start:end])
        start+=chunk_size-overlap
    return chunks

records=[]
for idx,row in df.iterrows():
    chunks=fixed_chunks(row['answer'])
    for chunk_id,chunk_text in enumerate(chunks):
        records.append({
            'chunk_text': chunk_text,
            'question': row['question'],
            'source': row['source'],
            'focus_area': row['focus_area'],
            'doc_id': idx,
            'chunk_id': chunk_id
        })

chunks_df=pd.DataFrame(records)
print(chunks_df.shape)
chunks_df.head()

(55426, 6)


,chunk_text,question,source,focus_area,doc_id,chunk_id
0,Glaucoma is a group of diseases that can damag...,What is (are) Glaucoma ?,NIHSeniorHealth,Glaucoma,0,0
1,nourishes the nearby tissues. (Watch the video...,What is (are) Glaucoma ?,NIHSeniorHealth,Glaucoma,0,1
2,rts of the eye and result in loss of vision. O...,What is (are) Glaucoma ?,NIHSeniorHealth,Glaucoma,0,2
3,essure inside the eye to build. If the pressur...,What is (are) Glaucoma ?,NIHSeniorHealth,Glaucoma,0,3
4,to learn more. See a glossary of glaucoma te...,What is (are) Glaucoma ?,NIHSeniorHealth,Glaucoma,0,4


In [9]:
from sentence_transformers import SentenceTransformer

embed_model=SentenceTransformer('all-MiniLM-L6-v2')

texts=chunks_df['chunk_text'].tolist()
embeddings=embed_model.encode(texts,batch_size=64,show_progress_bar=True,convert_to_numpy=True)
print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/867 [00:00<?, ?it/s]

(55426, 384)


In [10]:
import faiss
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))
print(index.ntotal)

55426


In [11]:
def retrieve(query,k=5):
    query_emb=embed_model.encode([query],convert_to_numpy=True).astype('float32')
    distances,indices=index.search(query_emb,k)
    results=chunks_df.iloc[indices[0]].copy()
    results["distance"]=distances[0]
    return results

results=retrieve("What lifestyle changes help hypertension?")
print(results[['chunk_text','source','distance']])

                                              chunk_text             source  \
10004  rains, fish, poultry, beans, seeds, nuts, and ...  MPlusHealthTopics   
10023  as stroke, heart failure, heart attack and kid...  MPlusHealthTopics   
31613  ges, try making one healthy lifestyle change a...              NHLBI   
31612  er related problems.\n                \n\n    ...              NHLBI   
31587  nd sodium sensitivity\n                \nDrink...              NHLBI   

       distance  
10004  0.712221  
10023  0.717616  
31613  0.719488  
31612  0.747197  
31587  0.754125  


In [12]:
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig

model_name='Qwen/Qwen2.5-3B-Instruct'

bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)

tokenizer=AutoTokenizer.from_pretrained(model_name)
llm=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto'
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [13]:
def build_prompt(query,retrieved_chunks):
    evidence_text=""
    for i,row in retrieved_chunks.iterrows():
        evidence_text+=f"[Source:{row['source']}]/n{row['chunk_text']}/n/n"

    prompt=f"""You are a medical information assistant.Answer the user's question usnig ONLY the evidence provided below.Do not use outside knowledge.If the evidence is insufficient to answer confidently,say so clearly.

    Evidence:{evidence_text}

    Question:{query}

    Answer (cite sources using [Source:name] where relevant):"""
    return prompt

def generate_answer(query,k=5,max_new_tokens=300):
    retrieved=retrieve(query,k=k)
    prompt=build_prompt(query,retrieved)

    messages=[{"role":"user","content":prompt}]
    text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tokenizer(text,return_tensors='pt').to(llm.device)

    output=llm.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False)
    response=tokenizer.decode(output[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True)

    return response,retrieved

answer,evidence=generate_answer("What lifestyle changes hypertension?")
print(answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The lifestyle changes that can help control high blood pressure include:

- **Healthy Eating**: This includes following the DASH diet, which emphasizes fruits, vegetables, and low-fat dairy products, while limiting sodium intake.
- **Being Physically Active**: Regular exercise can help reduce blood pressure.
- **Maintaining a Healthy Weight**: Being overweight or obese increases blood pressure. Losing weight can lower it.
- **Limiting Alcohol Intake**: Excessive alcohol consumption can lead to high blood pressure.
- **Managing Stress**: Techniques like meditation and yoga can help manage stress levels.

**Sources:**
- [Source:MPlusHealthTopics]
- [Source:NHBLI]
- [Source:NIDDK]
- [Source:NHLBI]


In [14]:
EMERGENCY_KEYWORDS=[
    "chest pain","can't breathe","cannot breathe","heart attack","stroke","overdose","unconscious","suicide","kill myself","severe bleeding","emergency","dying","call 911"
]

HARMFUL_KEYWORDS=[
    "how to harm","how to hurt","fake symptoms","drug to kill","poison","how to overdose"
]

def safety_check(query):
    query_lower=query.lower()

    for keyword in EMERGENCY_KEYWORDS:
        if keyword in query_lower:
            return{
                "safe":False,
                "type":"emergency",
                "message":"This sounds like a medical emergency.Please call emergency services(112,911) immediately or go to your nearest emergency room.DO not rely on this assistant for emergencies."
            }

    for keyword in HARMFUL_KEYWORDS:
        if keyword in query_lower:
            return{
                "safe":False,
                "type":"harmful",
                "message":"I'm unable to assist with this request.If you or someone you know needs help,please contact a medical professional."
            }

    return{"safe":True,"type":None,"message":None}

print(safety_check("I have chest pain what do I do"))
print(safety_check("What lifestyle changes help hypertension?"))

{'safe': False, 'type': 'emergency', 'message': 'This sounds like a medical emergency.Please call emergency services(112,911) immediately or go to your nearest emergency room.DO not rely on this assistant for emergencies.'}
{'safe': True, 'type': None, 'message': None}


In [15]:
def estimate_confidence(retrieved_chunks):
    distances=retrieved_chunks['distance'].values

    avg_distance=np.mean(distances)
    score=round(max(0,min(1,1-(avg_distance/1.5))),2)

    if score>=0.6:
        confidence="High"
    elif score>=0.4:
        confidence="Medium"
    else:
        confidence="Low"

    unique_sources=retrieved_chunks['source'].nunique()

    return{
        "confidence_label":confidence,
        "confidence_score":score,
        "avg_retrieval_distance":round(float(avg_distance),4),
        "unique_sources":unique_sources
    }

conf=estimate_confidence(evidence)
print(conf)

{'confidence_label': 'Medium', 'confidence_score': np.float32(0.48), 'avg_retrieval_distance': 0.7738, 'unique_sources': 3}


In [16]:
def query_assistant(query,k=5,max_new_tokens=300):
    safety=safety_check(query)
    if not safety["safe"]:
        return{
            "query":query,
            "answer":safety["message"],
            "confidence":None,
            "sources":None,
            "safety_triggered":safety["type"]
        }

    retrieved=retrieve(query,k=k)

    answer,_=generate_answer(query,k=k,max_new_tokens=max_new_tokens)

    confidence=estimate_confidence(retrieved)

    sources=retrieved[['source','focus_area','chunk_text']].to_dict(orient='records')

    return {
        "query": query,
        "answer": answer,
        "confidence": confidence,
        "sources": sources,
        "safety_triggered": None
    }

result = query_assistant("What are the symptoms of diabetes?")
print("Answer:\n", result["answer"])
print("\nConfidence:", result["confidence"])
print("\nSafety triggered:", result["safety_triggered"])

Answer:
 The symptoms of diabetes can vary widely, but some common signs include:

- Being very thirsty
- Frequent urination, especially at night
- Feeling very hungry
- Feeling very tired
- Losing weight without trying
- Sores that heal slowly
- Dry, itchy skin
- Loss of feeling or tingling in the feet
- Blurry eyesight

It's important to note that many people with diabetes may not experience any noticeable symptoms, and the only way to know for sure if you have diabetes is to have your doctor do a blood test. Approximately 29 million Americans age 20 or older have diabetes, according to 2014 estimates from the Centers for Disease Control and Prevention. 

[Source:NIDDK]  
[Source:NIHSeniorHealth]  
[Source:NIHSeniorHealth]  
[Source:NIDDK]

Confidence: {'confidence_label': 'High', 'confidence_score': np.float32(0.68), 'avg_retrieval_distance': 0.4839, 'unique_sources': 2}

Safety triggered: None


In [17]:
import requests
import re
from bs4 import BeautifulSoup

In [18]:
def fetch_webpage_text(url,source_name):
    response=requests.get(url,timeout=10)
    soup=BeautifulSoup(response.text,'html.parser')

    for tag in soup(['script','style','nav','footer','header']):
        tag.decompose()

    text=soup.get_text(separator='',strip=True)

    text=re.sub(r'\s+',' ',text).strip()
    return text

who_url="https://www.who.int/news-room/fact-sheets/detail/hypertension"
who_text=fetch_webpage_text(who_url,"WHO")

cdc_url = "https://www.cdc.gov/high-blood-pressure/about/index.html"
cdc_text=fetch_webpage_text(cdc_url,"CDC")

nice_url = "https://www.nice.org.uk/guidance/ng136/chapter/Recommendations"
nice_text=fetch_webpage_text(nice_url,"NICE")

print(len(who_text))
print(len(cdc_text))
print(len(nice_text))

9264
8434
33537


In [19]:
new_sources=[
    {"text":who_text,"source":"WHO","focus_area":"Hypertension"},
    {"text":cdc_text,"source":"CDC","focus_area":"High Blood Pressure"},
    {"text":nice_text,"source":"NICE","focus_area":"Hypertension Guidelines"}
]

new_records=[]
start_doc_id=chunks_df['doc_id'].max()+1

for i,src in enumerate(new_sources):
    chunks=fixed_chunks(src["text"])
    for chunk_id,chunk_text in enumerate(chunks):
        new_records.append({
            'chunk_text':chunk_text,
            'question':None,
            'source':src["source"],
            'focus_area':src["focus_area"],
            'doc_id':start_doc_id+i,
            'chunk_id':chunk_id
        })

new_chunks_df=pd.DataFrame(new_records)
print(new_chunks_df.shape)

new_embeddings=embed_model.encode(
    new_chunks_df['chunk_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

index.add(new_embeddings.astype('float32'))
chunks_df=pd.concat([chunks_df,new_chunks_df],ignore_index=True)

print(f"Total chunks: {len(chunks_df)}")
print(f"Total vectors in index: {index.ntotal}")

results=retrieve("What are the recommendations for treating hypertension?",k=5)
print(results[['source','focus_area','chunk_text']])

result=query_assistant("What are the treatment options for hypertension?")
print("Answer:\n",result["answer"])
print("\nConfidence:",result["confidence"])

(115, 6)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Total chunks: 55541
Total vectors in index: 55541
                source           focus_area  \
55435              WHO         Hypertension   
100    NIHSeniorHealth  High Blood Pressure   
55436              WHO         Hypertension   
99     NIHSeniorHealth  High Blood Pressure   
55442              WHO         Hypertension   

                                              chunk_text  
55435  ressure less than 140/90.There are several com...  
100    with blood pressure control. Follow a healthy ...  
55436  er from the body, lowering blood pressure.Prev...  
99     You can take steps to help prevent high blood ...  
55442  Health Organization (WHO) supports countries t...  
Answer:
 The treatment options for hypertension include:

- Medications:
  - ACE inhibitors: enalapril, lisinopril (relax blood vessels and prevent kidney damage)
  - Angiotensin-2 receptor blockers (ARBs): losartan, telmisartan (relax blood vessels and prevent kidney damage)
  - Calcium channel blockers: amlodi

In [20]:
import gradio as gr

def assistant_interface(query):
    if not query.strip():
        return "Please enter a question.","", ""

    result=query_assistant(query)

    if result["safety_triggered"]:
        return result["answer"],"N/A","⚠️ Safety filter triggered"

    answer=result["answer"]

    conf=result["confidence"]
    confidence_str=f"Label: {conf['confidence_label']} | Score: {conf['confidence_score']} | Avg Distance: {conf['avg_retrieval_distance']} | Unique Sources: {conf['unique_sources']}"

    sources=result["sources"]
    sources_str=""
    for s in sources:
        sources_str+=f"**Source:** {s['source']} | **Topic:** {s['focus_area']}\n"
        sources_str+=f"{s['chunk_text'][:200]}...\n\n"

    return answer,confidence_str,sources_str

with gr.Blocks(title="Healthcare Information Assistant") as demo:
    gr.Markdown("# 🏥 Healthcare Information Assistant")
    gr.Markdown("Ask health-related questions. Answers are grounded in trusted medical sources (MedQuAD,WHO,CDC,NICE).")

    with gr.Row():
        query_input=gr.Textbox(label="Your Question",placeholder="e.g. What are the symptoms of diabetes?",lines=2)

    submit_btn=gr.Button("Ask",variant="primary")

    with gr.Row():
        answer_output=gr.Markdown(label="Answer")

    with gr.Row():
        confidence_output=gr.Textbox(label="Confidence",interactive=False)

    with gr.Row():
        sources_output=gr.Markdown(label="Retrieved Evidence")

    submit_btn.click(
        fn=assistant_interface,
        inputs=[query_input],
        outputs=[answer_output,confidence_output,sources_output]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://4d1c910c2ddbcd1e28.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
